<a target="_blank" href="https://colab.research.google.com/github/Tensor-Reloaded/Neural-Networks-Template-2025/blob/main/Lab02/Assignment1.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# **Assignment 1 (10 points)**

## **Solving a linear system in python**

In this homework, you will familiarize yourself with key linear algebra con-
cepts and Python programming by solving a system of linear equations. You
will explore multiple methods for solving such systems, including Cramer’s rule
and matrix inversion. By the end of this assignment, you will have a good un-
derstanding of how to represent and manipulate matrices and vectors in Python.

We begin with the following system of 3 linear equations with 3 unknowns:
$$ 2x + 3y - z = 5 $$
$$ x - y + 4z = 6 $$
$$ 3x + y + 2z = 7 $$

This system can be vectorized in the following form:
$$ A \cdot X = B $$
where:
$$
A = \begin{bmatrix}
2 & 3 & -1 \\
1 & -1 & 4 \\
3 & 1 & 2
\end{bmatrix}, \quad 
X = \begin{bmatrix}
x \\
y \\
z
\end{bmatrix}, \quad 
B = \begin{bmatrix}
5 \\
6 \\
7
\end{bmatrix}
$$

**Considerations**
- do not use any linear algebra framework such as $numpy$
- use python lists as data structures for matrices and vectors
- experiment with other values for the coefficients and free terms

### **1. Parsing the System of Equations (1 point)**

The first task is to implement a Python script that reads a system of linear equations from a text file and parses it into a matrix $A$ and a vector $B$. You will use the input format described below to extract the coefficients for $A$ and $B$.

**Input File Format**
```text
2x + 3y - z = 5
x - y + 4z = 6
3x + y + 2z = 7
```

Note that the coefficients are always in the order x, y and z and the terms are always space separated

In [558]:
import pathlib

def load_system(path: pathlib.Path) -> tuple[list[list[float]], list[float]]:
    A = []
    B = []
    with open(path) as fin:
        while len(tokens := list(fin.readline().split())):
            sgn = 1.0
            row = []
            in_vector = False
            for token in tokens:
                if token == '+':
                    sgn = 1.0
                elif token == '-':
                    sgn = -1.0
                elif token == '=':
                    in_vector = True
                    sgn = 1.0
                else:
                    if in_vector:
                        B.append(sgn*float(token))
                    else:
                        row.append(sgn*(float(token[:-1]) if len(token)>1 else 1))
            A.append(row)         
    return A, B


A, B = load_system(pathlib.Path("system.txt"))
print(f"{A=} {B=}")

A=[[1.0, 0.0, 1.0], [1.0, -2.0, 0.0], [3.0, 0.0, 5.0]] B=[-5.0, 3.0, -7.0]


### **2. Matrix and Vector Operations (5 points)**

Once you have successfully parsed the matrix and vector, complete the following exercises to manipulate and understand basic matrix and vector operations. Write Python functions for each of these tasks:

#### 2.1. Determinant

Write a function to compute the determinant of matrix $A$. Recall one of the formulae for the determinant of a $3x3$ matrix:
$$ \text{det}(A) = a_{11}(a_{22}a_{33} - a_{23}a_{32}) - a_{12}(a_{21}a_{33} - a_{23}a_{31}) + a_{13}(a_{21}a_{32} - a_{22}a_{31}) $$

In [559]:
def iszero(x: float) -> bool:
    return abs(x)<=1e-12

def determinant(matrix: list[list[float]]) -> float: # facut cu eliminare gaussiana ca sa mearga si pentru matrici mai mari
    b = [row[:] for row in matrix]  # ca sa nu modific matricea initiala
    n = len(matrix)
    m = len(matrix[0])
    if n != m:
        return 0
    det = 1.0
    for j in range(m):
        for i in range(j,n):
            if not iszero(b[i][j]):
                if i != j:
                    b[i], b[j] = b[j], b[i]
                    det = -det
                for i2 in range(n):
                    if j == i2 or iszero(b[i2][j]): continue
                    coef = b[j][j]/b[i2][j]
                    det /= coef
                    for j2 in range(m):
                        b[i2][j2] = b[i2][j2]*coef - b[j][j2]
                break
    for i in range(min(n,m)):
        det *= b[i][i]
    return det

print(f"{determinant(A)=}")

determinant(A)=-3.999999999999999


#### 2.2. Trace

Compute the sum of the elements along the main diagonal of matrix $A$. For a matrix $A$, this is:
$$ \text{Trace}(A) = a_{11} + a_{22} + a_{33} $$

In [560]:
def trace(matrix: list[list[float]]) -> float:
    tr = 0
    for i in range(min(len(matrix),len(matrix[0]))):
        tr += matrix[i][i]
    return tr

print(f"{trace(A)=}")

trace(A)=4.0


#### 2.3. Vector norm

Compute the Euclidean norm of vector $B$, which is:
$$ ||B|| = \sqrt{b_1^2 + b_2^2 + b_3^2} $$

In [561]:
from math import sqrt
def norm(vector: list[float]) -> float:
    s = 0
    for x in vector:
        s += x*x
    return sqrt(s)

print(f"{norm(B)=}")

norm(B)=9.1104335791443


#### 2.4. Transpose of matrix

Write a function to compute the transpose of matrix $A$. The transpose of a matrix $A$ is obtained by swapping its rows and columns.
    

In [562]:
def transpose(matrix: list[list[float]]) -> list[list[float]]:
    return [[matrix[i][j] for i in range(len(matrix))] for j in range(len(matrix[0]))]

print(f"{transpose(A)=}")

transpose(A)=[[1.0, 1.0, 3.0], [0.0, -2.0, 0.0], [1.0, 0.0, 5.0]]


#### 2.5. Matrix-vector multiplication

Write a function that multiplies matrix $A$ with vector $B$.

In [563]:
def multiply(matrix: list[list[float]], vector: list[float]) -> list[float] or None:
    if matrix is None or vector is None:
        return None
    n = len(matrix)
    m = max(len(vector), len(matrix[0])) # nXm * mX1
    output = [0.0]*n
    for i in range(n):
        for j in range(m):
            output[i] += (matrix[i][j] if j < len(matrix[i]) else 0) * (vector[j] if j < len(vector) else 0)
    return output


print(f"{multiply(A, B)=}")

multiply(A, B)=[-12.0, -11.0, -50.0]


### **3. Solving using Cramer's Rule (1 point)**

Now that you have explored basic matrix operations, solve the system of linear equations using Cramer's rule.

**Cramer's Rule:**

Cramer's rule allows you to solve for each unknown $x$, $y$, and $z$ using determinants. For example:
$$ x = \frac{\text{det}(A_x)}{\text{det}(A)}, \quad y = \frac{\text{det}(A_y)}{\text{det}(A)}, \quad z = \frac{\text{det}(A_z)}{\text{det}(A)} $$
where $A_x$, $A_y$, and $A_z$ are matrices formed by replacing the respective column of matrix $A$ with vector $B$.

In [564]:
def solve_cramer(matrix: list[list[float]], vector: list[float]) -> list[float] or None:
    n = len(matrix)
    m = len(matrix[0])
    ans = [0]*m
    orig = [0]*n 
    d = determinant(matrix)

    for j in range(m):
        for i in range(n):
            orig[i]=matrix[i][j]
            matrix[i][j]=vector[i]

        d1 = determinant(matrix)
        if d != 0:
            ans[j] = d1/d
        else:
            if d1 == 0:
                ans[j] = 0 # poate fi orice
            else:
                return None

        for i in range(n):
            matrix[i][j]=orig[i]

    return ans

def check_cramer():
    result = solve_cramer(A, B)
    f"solve_cramer(A,B)={result}"
    if result is not None:
        print(multiply(A, result), B, sep='\n')
check_cramer()

[-5.0, 3.0, -7.0]
[-5.0, 3.0, -7.0]


### **4. Solving using Inversion (3 points)**

Finally, solve the system by computing the inverse of matrix $A$ and multiplying it by vector $B$.
$$ A \cdot X = B \rightarrow X = A^{-1} \cdot B $$
**Adjugate Method for Matrix Inversion:**

To find the inverse of matrix $ A $, you can use the adjugate method:
$$ A^{-1} = \frac{1}{\text{det}(A)} \times \text{adj}(A) $$
where $\text{adj}(A)$ is the adjugate (or adjoint) matrix, which is the transpose of the cofactor matrix of $ A $.

**Cofactor Matrix:**

The cofactor matrix is a matrix where each element is replaced by its cofactor. The cofactor of an element $a_{ij}$ is given by:
$$ (-1)^{i+j} \times \text{det}(M_{ij}) $$
where $M_{ij}$ is the minor of element $a_{ij}$, which is the matrix obtained by removing the $i$-th row and $j$-th column from matrix $A$.

In [ ]:
def minor(matrix: list[list[float]], i: int, j: int) -> list[list[float]]:
    n=len(matrix)
    m=len(matrix[0])
    return [
        [
            matrix[line][col] for col in range(m) if col != j
        ] for line in range(n) if line != i
    ]

def cofactor(matrix: list[list[float]]) -> list[list[float]]:
    n=len(matrix)
    m=len(matrix[0])
    return [
        [
            (-1 if (i+j)%2 else 1)*determinant(minor(matrix,i,j)) for j in range(m)
        ] for i in range(n)
    ] 

def adjoint(matrix: list[list[float]]) -> list[list[float]]:
    return transpose(cofactor(matrix))

def inverse(matrix: list[list[float]]) -> list[list[float]] or None:
    n=len(matrix)
    m=len(matrix[0])
    d=determinant(matrix)
    if d == 0:
        return None
    adj=adjoint(matrix)
    return [
        [
            adj[i][j]/d for j in range(n)
        ] for i in range(m)
    ] 
def solve(matrix: list[list[float]], vector: list[float]) -> list[float]:
    return multiply(inverse(matrix), vector)

def check_adjoint():
    result = solve(A, B)
    print(f"solve(A,B)={result}")
    if result is not None:
        print(multiply(A, result), B, sep='\n')
check_adjoint()

[[-10.0, -5.0, 6.0], [-0.0, 1.9999999999999996, -0.0], [2.0, 1.0, -2.0]]
solve(A,B)=[-9.0, -6.0, 4.000000000000001]
[-4.999999999999999, 3.0, -6.9999999999999964]
[-5.0, 3.0, -7.0]


### **(Bonus) 5. Solving using Gaussian elimination**

In [566]:
def solve_gauss(matrix: list[list[float]], vector: list[float]) -> list[float] or None:
    b = [row[:] for row in matrix]  # ca sa nu modific matricea initiala
    val = vector[:]
    n = len(matrix)
    m = len(matrix[0])
    ans = [0]*m
    found_vars = 0
    for j in range(m):
        for i in range(found_vars, n):
            if not iszero(b[i][j]):
                if i != found_vars:
                    b[i], b[found_vars] = b[found_vars], b[i]
                    val[i], val[found_vars] = val[found_vars], val[i]
                for i2 in range(n):
                    if found_vars == i2 or iszero(b[i2][j]): continue
                    coef = b[found_vars][j]/b[i2][j]
                    for j2 in range(m):
                        b[i2][j2] = b[i2][j2]*coef - b[found_vars][j2]
                    val[i2]=val[i2]*coef - val[found_vars]
                found_vars += 1
                break
    for i in range(n):
        for j in range (m):
            if not iszero(b[i][j]):
                ans[j] = val[i]/b[i][j] 
                val[i] = 0 
        if not iszero(val[i]):
            return None # avem o ecuatie de tipul 0*ans[1]+...+0*ans[m]=x, x nenul
    return ans


def check_gauss():
    result = solve_gauss(A, B)
    print(f"solve_gauss(A,B)={result}")
    if(result):
        print(multiply(A, result), B, sep='\n')
check_gauss()

solve_gauss(A,B)=[-9.000000000000002, -6.000000000000001, 4.000000000000002]
[-5.0, 3.0, -7.0]
[-5.0, 3.0, -7.0]
